# Daedalus Devs Team Report # 1 (03/24/2026)

### What we have done this week:
* Installed libraries and dependencies needed **(cell 1)**
* Figured out how to upload resume PDF file to Colab and use it with the LLM
* Created a function that takes in user's resume and parse it to output the extracted info **(cell 3)**
* Created a function that uses Skyvern, played around a little to see how it works. **(cell 4)**

### Plans for next week:
* Need to figure out how to see the filled in website, not as a screenshot but in another tab.
* Follow our workflow where the user can make corrections to the extracted resume info.

![Workflow Diagram](/Users/sanilachowdhury/Desktop/ai_agents/job-agent/AI_AGENT_WORKFLOW.png)

In [ ]:

%pip install chromadb
# %pip install "skyvern==1.0.24"
%pip install skyvern
# %playwright install --with-deps chromium
# %playwright install chromium
%pip install langchain langchain-huggingface
%pip install python-dotenv

print ("Done installing")

In [ ]:
#installing dependencies for browser-use (an alternative to skyvern)
%pip install browser-use playwright
%pip install -U langchain-google-genai
%pip install -U browser-use
# %playwright install chromium

In [7]:
# import os
# from dotenv import load_dotenv


# load_dotenv()


# gemini_key = os.getenv("GEMINI_API_KEY")
# skyvern_key = os.getenv("SKYVERN_API_KEY")
# phil_key = os.getenv("PHIL-KEY")


# os.environ["ENABLE_AUTH"] = "false"

# # from skyvern.forge.sdk import SkyvernForge
# from skyvern import Skyvern
# import nest_asyncio
# import asyncio

# # Apply nest_asyncio to allow nested event loops in Colab
# nest_asyncio.apply()

# # 2. Map it to the environment variables Skyvern/LiteLLM looks for
# os.environ["GEMINI_API_KEY"] = gemini_key
# # os.environ["GOOGLE_API_KEY"] = gemini_key  # Some versions prefer this
# # Optional: Force Skyvern to use a specific model
# os.environ["LLM_KEY"] = "gemini/gemini-1.5-flash"
# print(f"Key check: {'FOUND' if os.environ.get('GEMINI_API_KEY') else 'MISSING'}")
# # print("Keys loaded and environment configured.")


import os
from dotenv import load_dotenv
import platform
import sys


#Sanila needs to set the working directory explicity (macOS) but it's not necessary on Windows or WSL.
#added this macro so we don't have to manually comment/uncommment os.chdir() every time.

IS_MAC = platform.system() == "Darwin"

if IS_MAC:
    os.chdir("/Users/sanilachowdhury/Desktop/ai_agents")
    print("Running on macOS")
else:
    print("Not running on macOS, current directory:", os.getcwd())

# load .env and show where it's looking
load_dotenv()
print(f"Looking for .env in: {os.getcwd()}")
print(f"GEMINI_API_KEY found: {bool(os.getenv('GEMINI_API_KEY'))}")

gemini_key = os.getenv("GEMINI_API_KEY")
skyvern_key = os.getenv("SKYVERN_API_KEY")
phil_key = os.getenv("PHIL-KEY")

if not gemini_key:
    raise ValueError("GEMINI_API_KEY not found! Check that .env is in: " + os.getcwd())

os.environ["ENABLE_AUTH"] = "false"

from skyvern import Skyvern
import nest_asyncio
import asyncio

nest_asyncio.apply()

os.environ["GEMINI_API_KEY"] = gemini_key
os.environ["LLM_KEY"] = "gemini/gemini-2.0-flash-lite"
print(f"Key check: {'FOUND' if os.environ.get('GEMINI_API_KEY') else 'MISSING'}")

Not running on macOS, current directory: /home/rb4271/ai-agents/job-agent
Looking for .env in: /home/rb4271/ai-agents/job-agent
GEMINI_API_KEY found: True
Key check: FOUND


In [ ]:
url = "https://demo.applitools.com/"

In [ ]:
#NOTE: This uses browser-use instead of skyvern. This cell works but it does not promp the user for addtional info when needed. See next cell.
import os
import asyncio
from dotenv import load_dotenv
from browser_use import Agent, BrowserProfile, BrowserSession
from browser_use.llm import ChatGoogle

load_dotenv()

url = "https://demo.applitools.com/"

async def run_locally():
    llm = ChatGoogle(
        model="gemini-2.5-flash-lite",
        api_key=os.getenv("GEMINI_API_KEY")
    )

    browser_profile = BrowserProfile(headless=False, keep_alive=True)
    browser_session = BrowserSession(browser_profile=browser_profile)

    agent = Agent(
        task=f"Go to {url} and fill in 'test_user' and 'password123' BUT DO NOT PRESS SUBMIT OR LOGIN. Return DONE when complete",
        llm=llm,
        browser_session=browser_session,
    )

    result = await agent.run()
    print(result)

await run_locally()

In [ ]:
from browser_use import Agent, BrowserProfile, BrowserSession
from browser_use.llm import ChatGoogle

async def run_job_application():
    llm = ChatGoogle(model="gemini-2.5-flash-lite", api_key=os.getenv("GEMINI_API_KEY"))
    
    browser_profile = BrowserProfile(headless=False, keep_alive=True)
    browser_session = BrowserSession(browser_profile=browser_profile)

    # Phase 1: fill what we know
    agent = Agent(
        task=f"""Go to {url} and fill in the username with a_cool_username_123. 
        If you encounter fields you don't have info for, stop and list what's missing. 
        Do NOT submit.""",
        llm=llm,
        browser_session=browser_session,
        use_vision=False,  # reduces tokens significantly
    )
    result = await agent.run()
    print("Agent stopped. Result:", result)


    #NOTE: Might be better to put this in a second cell or loop this?
    # Phase 2: ask user for missing info
    missing = input("What additional info do you want to provide? ")

    # Phase 3: continue with same browser session (no reload, no extra screenshots)
    agent2 = Agent(
        task=f"Fill in the remaining fields with: {missing}. Do NOT submit.",
        llm=llm,
        browser_session=browser_session,  # reuse same session!
    )
    result2 = await agent2.run()
    print("Done:", result2)

await run_job_application()

In [ ]:
# !skyvern quickstart
# from skyvern import Skyvern
# import asyncio

# async def run_locally():
#   client = Skyvern.local()
#   browser = await client.launch_local_browser()
#   page = await browser.get_working_page()
#   await page.goto(url)
#   await page.act("My username is user101 and my password is hello123. Please fill in those fields but DO NOT press submit/login")
#   print("page filled in!\n")

#   await browser.close()

# await run_locally()

from skyvern import Skyvern
import asyncio
import nest_asyncio
from google import genai
# from google.colab import userdata

nest_asyncio.apply()

client_genai = genai.Client(api_key=gemini_key)

# this takes in user's resume and parse and LLM will output the info it extracted to ensure accuracy
def parse_pdf_with_gemini(file_path, prompt="Please parse this resume and return its contents in a structured JSON that is easy for both LLM and humans to read."):
    """
    Uploads a PDF to Gemini and returns the model's analysis.
    """
    # upload user's resume
    print(f"Uploading {file_path}...")
    uploaded_file = client_genai.files.upload(
        file=file_path,
        config={"display_name": "User_PDF"}
    )

   # feeds LLM the prompt and uploaded resume PDF
    response = client_genai.models.generate_content(
        # model="gemini-2.0-flash",
        model="gemini-2.5-flash-lite",
        contents=[uploaded_file, prompt]
    )

    return response.text

# check if agent extracted the resume's info correctly
def verify_resume_info(extracted_info):
    """
    Shows the extracted info to the user and allows them to correct it.
    Keeps looping until the user confirms it's correct.
    """
    current_info = extracted_info

    while True:
        print("\n" + "="*10)
        print("EXTRACTED RESUME INFO:")
        print("="*10)
        print(current_info)
        print("="*10)

        user_confirm = input("\nIs this information correct? (y/n): ").strip().lower()

        if user_confirm == "y":
            print("\nGreat! Moving forward with this information.")
            return current_info
        else:
            correction = input("\nWhat needs to be corrected or added? Please describe: ").strip()

            print("\nUpdating extracted info...")
            response = client_genai.models.generate_content(
                model="gemini-2.5-flash-lite",
                contents=[
                    f"""Here is the currently extracted resume information:
{current_info}

The user wants the following correction or addition:
{correction}

Please update the extracted resume JSON accordingly and return the full updated JSON."""
                ]
            )
            current_info = response.text
            print("\nInfo updated! Let's review it again.")


# call the function to store the extracted info
result = parse_pdf_with_gemini("content/jakes-resume.pdf")

# run verify_resume_info to verify and correct
final_info = verify_resume_info(result)

print("\nFinal verified resume info:")
print(final_info)
# call the function to store the extracted info and print it
# result = parse_pdf_with_gemini("content/jakes-resume.pdf")
# print(result)




# Plans for next week:
- Need to figure out how to see the filled in website, not as a screenshot but in another tab.
- Follow our workflow where the user can make corrections to the extracted resume info.
- put agent graph at the top of notebook for next week